# Load all three files

In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np

In [2]:
# ── Cell 2: Load the files ───────────────────────────────────────
players     = pd.read_csv('../data/players.csv')
appearances = pd.read_csv('../data/appearances.csv')

print("players shape:    ", players.shape)
print("appearances shape:", appearances.shape)

players shape:     (47702, 26)
appearances shape: (1862208, 13)


# Understand what's in each file

In [3]:
# ── Cell 3: Inspect players ──────────────────────────────────────
players.head(3)

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0


In [4]:
# ── Cell 4: Inspect appearances ──────────────────────────────────
appearances.head(3)

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45


In [5]:
# ── Cell 5: Check missing values ─────────────────────────────────
print("=== players missing ===")
print(players.isnull().sum())

print("\n=== appearances missing ===")
print(appearances.isnull().sum())

=== players missing ===
player_id                                   0
first_name                               3095
last_name                                   0
name                                        0
last_season                                 0
current_club_id                             0
player_code                                 0
country_of_birth                         5162
city_of_birth                            4887
country_of_citizenship                    271
date_of_birth                              49
sub_position                              455
position                                    0
foot                                     5256
height_in_cm                             3756
contract_expiration_date                16488
agent_name                              22180
image_url                                   0
international_caps                      29979
international_goals                     29979
current_national_team_id                44611
url       

# Aggregate appearances into per-player totals

In [6]:
# ── Cell 6: Aggregate appearances → one row per player ───────────
# appearances has one row per match — collapse to season totals
player_stats = appearances.groupby('player_id').agg(
    games_played       = ('appearance_id',  'count'),
    total_goals        = ('goals',          'sum'),
    total_assists      = ('assists',         'sum'),
    total_minutes      = ('minutes_played',  'sum'),
    total_yellow_cards = ('yellow_cards',    'sum'),
    total_red_cards    = ('red_cards',       'sum'),
).reset_index()

print(player_stats.shape)
player_stats.head()

(28665, 7)


,player_id,games_played,total_goals,total_assists,total_minutes,total_yellow_cards,total_red_cards
0,10,136,48,25,8808,19,0
1,26,152,0,0,13508,4,2
2,65,122,38,13,8788,11,1
3,77,4,0,0,307,0,0
4,80,12,0,0,1080,0,0


# Join everything together with merge

In [7]:
# ── Cell 7: Merge players + stats ────────────────────────────────
# players already has market_value_in_eur — no need to join valuations
df = players.merge(player_stats, on='player_id', how='left')

print("After merge:", df.shape)
df.head(3)

After merge: (47702, 32)


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur,games_played,total_goals,total_assists,total_minutes,total_yellow_cards,total_red_cards
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0,136.0,48.0,25.0,8808.0,19.0,0.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,L1,Borussia Dortmund,750000.0,8000000.0,152.0,0.0,0.0,13508.0,4.0,2.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0,122.0,38.0,13.0,8788.0,11.0,1.0


# Keep only useful columns

In [8]:
# ── Cell 8: Keep only useful columns ─────────────────────────────
cols_to_keep = [
    'player_id',
    'name',
    'position',
    'sub_position',
    'foot',
    'height_in_cm',
    'country_of_citizenship',
    'current_club_name',
    'current_club_domestic_competition_id',
    'market_value_in_eur',
    'highest_market_value_in_eur',
    'games_played',
    'total_goals',
    'total_assists',
    'total_minutes',
    'total_yellow_cards',
    'total_red_cards',
]

df = df[cols_to_keep].copy()
print(df.shape)

(47702, 17)


# Filter to players with real playtime

In [9]:
# ── Cell 9: Filter to players with real playtime ─────────────────
# Drop players who barely played — they'll pollute the clustering
df = df[df['total_minutes'] >= 90].copy()
print(f"After minutes filter: {df.shape[0]} players")

After minutes filter: 25088 players


# Handle missing values

In [10]:
# ── Cell 10: Handle missing values ───────────────────────────────
# Check what's still missing after merge
print(df.isnull().sum())

# Fill missing numeric stats with 0 (player appeared but no stats recorded)
stat_cols = ['total_goals','total_assists','total_minutes',
             'total_yellow_cards','total_red_cards','games_played']
df[stat_cols] = df[stat_cols].fillna(0)

# Drop rows missing position — we can't cluster without it
df.dropna(subset=['position'], inplace=True)

# Fill missing market value with median (rather than dropping)
df['market_value_in_eur'].fillna(df['market_value_in_eur'].median(), inplace=True)
df['highest_market_value_in_eur'].fillna(df['highest_market_value_in_eur'].median(), inplace=True)

# Fill height with median
df['height_in_cm'].fillna(df['height_in_cm'].median(), inplace=True)

print("\nAfter cleaning:")
print(df.isnull().sum())

player_id                                 0
name                                      0
position                                  0
sub_position                             44
foot                                    834
height_in_cm                            547
country_of_citizenship                  165
current_club_name                       582
current_club_domestic_competition_id    582
market_value_in_eur                     536
highest_market_value_in_eur             536
games_played                              0
total_goals                               0
total_assists                             0
total_minutes                             0
total_yellow_cards                        0
total_red_cards                           0
dtype: int64

After cleaning:
player_id                                 0
name                                      0
position                                  0
sub_position                             44
foot                                    834
he

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1612\499276111.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['market_value_in_eur'].fillna(df['market_value_in_eur'].median(), inplace=True)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1612\499276111.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or

# Fix data types

In [11]:
# ── Cell 11: Fix data types ───────────────────────────────────────
# Make sure numeric columns are actually numeric
df['market_value_in_eur']         = pd.to_numeric(df['market_value_in_eur'],         errors='coerce')
df['highest_market_value_in_eur'] = pd.to_numeric(df['highest_market_value_in_eur'], errors='coerce')
df['height_in_cm']                = pd.to_numeric(df['height_in_cm'],                errors='coerce')

df.dtypes

player_id                                 int64
name                                        str
position                                    str
sub_position                                str
foot                                        str
height_in_cm                            float64
country_of_citizenship                      str
current_club_name                           str
current_club_domestic_competition_id        str
market_value_in_eur                     float64
highest_market_value_in_eur             float64
games_played                            float64
total_goals                             float64
total_assists                           float64
total_minutes                           float64
total_yellow_cards                      float64
total_red_cards                         float64
dtype: object

# Final check

In [12]:
# ── Cell 12: Final check ──────────────────────────────────────────
print(f"Final dataset: {df.shape[0]} players, {df.shape[1]} columns")
print(f"\nPosition breakdown:")
print(df['position'].value_counts())
print(f"\nSample market values:")
print(df['market_value_in_eur'].describe())

Final dataset: 25088 players, 17 columns

Position breakdown:
position
Defender      8351
Midfield      7303
Attack        7157
Goalkeeper    2233
Missing         44
Name: count, dtype: int64

Sample market values:
count    2.455200e+04
mean     2.206173e+06
std      7.386093e+06
min      1.000000e+04
25%      1.500000e+05
50%      3.500000e+05
75%      1.000000e+06
max      2.000000e+08
Name: market_value_in_eur, dtype: float64


# Save

In [13]:
# ── Cell 13: Save ─────────────────────────────────────────────────
df.to_csv('../data/players_clean.csv', index=False)
print("Saved to players_clean.csv")

Saved to players_clean.csv
